# Modern Visualization Builder

**Purpose:** Export data as JSON and create professional JavaScript/HTML charts

---

## 1. Load & Prepare Data

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Load data
df = pd.read_csv('data/combined-scoring.csv')

# Convert numeric columns
numeric_cols = ['demo_yrs', 'bisbas_bas', 
                'bisbas_bis', 'susd_depression', 'susd_mania',
                'prob_gam']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Create readable labels
df['population'] = df['source'].map({
    'ad_data': 'Adolescent/Young Adult',
    'dvs_data': 'Older Adult',
    'g25_data': 'General Population'
})

df['gender_label'] = df['demo_gender'].map({'1': 'Male', '2': 'Female', '3': 'Other'})

print(f"✓ Loaded {len(df):,} participants")
print(f"\nPopulations: {df['population'].value_counts().to_dict()}")

✓ Loaded 2,560 participants

Populations: {'General Population': 1009, 'Adolescent/Young Adult': 855, 'Older Adult': 696}


## 2. Export Data for Visualizations

In [3]:
# Create visualizations directory
Path('visualizations').mkdir(exist_ok=True)

# Export age data by population
age_data = {
    'all': df['demo_yrs'].dropna().tolist(),
    'adolescent': df[df['population'] == 'Adolescent/Young Adult']['demo_yrs'].dropna().tolist(),
    'older': df[df['population'] == 'Older Adult']['demo_yrs'].dropna().tolist(),
    'general': df[df['population'] == 'General Population']['demo_yrs'].dropna().tolist()
}

with open('visualizations/age_data.json', 'w') as f:
    json.dump(age_data, f)

print("Exported age_data.json")
for key, values in age_data.items():
    print(f"  {key}: n={len(values):,}")

Exported age_data.json
  all: n=2,560
  adolescent: n=855
  older: n=696
  general: n=1,009


## 3. Create HTML Visualization

In [4]:
html_template = '''
<!DOCTYPE html>
<html>
<head>
    <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
    <style>
        body {
            margin: 0;
            padding: 20px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
            min-height: 100vh;
        }
        .card {
            max-width: 900px;
            margin: 0 auto;
            background: white;
            border-radius: 20px;
            padding: 40px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.2);
        }
        .card-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 30px;
            padding-bottom: 20px;
            border-bottom: 2px solid #f0f0f0;
        }
        .card-title {
            font-size: 28px;
            font-weight: 700;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
        }
        .btn-group {
            display: inline-flex;
            background: #f5f5f5;
            border-radius: 10px;
            padding: 4px;
        }
        .btn-group button {
            padding: 8px 20px;
            border: none;
            background: transparent;
            border-radius: 8px;
            font-size: 13px;
            font-weight: 600;
            color: #666;
            cursor: pointer;
            transition: all 0.2s;
        }
        .btn-group button.active {
            background: white;
            color: #667eea;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }
    </style>
</head>
<body>
<div class="card">
    <div class="card-header">
        <div class="card-title">Age Distribution</div>
        <div class="btn-group">
            <button class="active" onclick="selectPopulation('all', this)">All</button>
            <button onclick="selectPopulation('adolescent', this)">Adolescent</button>
            <button onclick="selectPopulation('older', this)">Older</button>
            <button onclick="selectPopulation('general', this)">General</button>
        </div>
    </div>
    <div id="myChart"></div>
</div>
<script>
    const colors = {
        all: '#667eea',
        adolescent: '#764ba2',
        older: '#4facfe',
        general: '#f093fb'
    };
    let data = {};
    let currentPopulation = 'all';

    fetch('visualizations/age_data.json')
        .then(response => response.json())
        .then(loadedData => {
            data = loadedData;
            updateChart();
        });

    function selectPopulation(population, button) {
        document.querySelectorAll('.btn-group button').forEach(btn => btn.classList.remove('active'));
        button.classList.add('active');
        currentPopulation = population;
        updateChart();
    }

    function updateChart() {
        const trace = {
            x: data[currentPopulation],
            type: 'histogram',
            nbinsx: 20,
            marker: { color: colors[currentPopulation], line: { color: 'white', width: 0.5 } },
            opacity: 0.9
        };
        const layout = {
            xaxis: { title: 'Age', gridcolor: 'rgba(0,0,0,0.05)' },
            yaxis: { title: 'Count', gridcolor: 'rgba(0,0,0,0.05)' },
            plot_bgcolor: 'white',
            paper_bgcolor: 'white',
            margin: { l: 60, r: 30, t: 20, b: 50 }
        };
        Plotly.react('myChart', [trace], layout, { responsive: true, displayModeBar: false });
    }
</script>
</body>
</html>
'''

with open('age_distribution.html', 'w') as f:
    f.write(html_template)

print("✓ Created age_distribution.html")
print("\n📊 Open age_distribution.html in your browser to see the chart!")

✓ Created age_distribution.html

📊 Open age_distribution.html in your browser to see the chart!


## 4. Quick Data Export Functions

In [ ]:
def export_by_category(variable, category_col, filename):
    """Export a variable split by category"""
    result = {}
    for category in df[category_col].dropna().unique():
        values = df[df[category_col] == category][variable].dropna().tolist()
        result[str(category)] = values
    
    with open(f'visualizations/{filename}', 'w') as f:
        json.dump(result, f)
    print(f"✓ Exported {filename}")
    return result

# Example: Depression scores by gambling status
# export_by_category('susd_depression', 'prob_gam', 'depression_data.json')

In [ ]:
def export_summary_stats(group_by, metric_col, filename):
    """Export grouped statistics for bar charts"""
    result = []
    for group in df[group_by].dropna().unique():
        subset = df[df[group_by] == group][metric_col].dropna()
        if len(subset) > 0:
            result.append({
                'category': str(group),
                'mean': round(subset.mean(), 2),
                'n': len(subset)
            })
    
    with open(f'visualizations/{filename}', 'w') as f:
        json.dump(result, f, indent=2)
    print(f"✓ Exported {filename}")
    return result

# Example: Mean age by population
# export_summary_stats('population', 'demo_yrs', 'age_summary.json')

## 5. Your Custom Visualizations

Use these cells to create your own data exports and charts.

In [ ]:
# Export your custom data here


In [ ]:
# Create your custom HTML visualization here
